<a href="https://colab.research.google.com/github/Civio-Creative/sample_outputs/blob/main/AI_Bootcamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Extraction from Sales Emails: An Eval Case Study

**Author:** Franchesca Thompson · AI Bootcamp 2026, Testing track

**What this is:** A structured-output extraction pipeline for inbound sales email, and the evaluation I built to find out whether its output can be trusted.

**Key findings from the first version**

- **The model invented a year.** Given "get back to me by 12/15," it returned `2023-12-15`, even though the prompt said to skip ambiguous dates.
- **The same email gave different answers on different runs.** The forwarded-chain email drifted between runs. The exported JSON and the notebook output disagreed on the same input.
- **Consistency checks alone hid real errors.** Emails that came back identical on every run still missed two requested actions and a sender name. Consistent does not mean correct.

**Results (v1 → v2, 3 runs per email)**

- Overall field accuracy: **86% → 94%**
- Emails consistent across runs: **3/4 → 4/4**
- Emails with every field correct on every run: **1/4 → 3/4**
- Cost per email: **+31%** (longer prompt), about $0.003 → $0.004
- **One regression:** the new company rule made the model return a raw domain (`bluetechlabs.com`) where v1 had the right answer. One E4 action is still missed. Both are covered in Section 8b.

**What I changed:** I set temperature to 0, passed the email's received date into the prompt, wrote explicit rules for quoted text and for how questions relate to actions, and tightened the schema field definitions. Then I re-ran the same test set against hand-labeled answers. Before/after results are in Section 9.

## 1. Problem

A sales inbox gets unstructured email: new inquiries, forwarded chains, contract replies, and vague follow-ups. Reps re-key the useful parts into the CRM by hand: who is writing, which company, dollar amounts, deadlines, and what they are asking for.

**Goal:** Pull those fields out automatically, reliably enough that some can go straight into the CRM and the rest can be queued for a quick human check.

**The question this notebook answers:** Which fields can be trusted, and what does it take to get them there?

## 2. Setup
Requires an `OPENAI_API_KEY` saved in Colab Secrets (the key icon in the left sidebar).

In [2]:
import json
import re
from statistics import mean
from typing import Optional

import pandas as pd
from google.colab import userdata
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

MODEL = "gpt-4o-2024-08-06"   # pinned snapshot so results are reproducible
N_RUNS = 3                     # runs per email, used to measure consistency

## 3. Schema decisions

**Schema discipline:** Every field is either lifted directly from the text or trivially derived from it. There is no subjective classification here (urgency, sentiment, intent). Those belong in a separate pass, where they can be evaluated on their own terms. Mixing them in would make it impossible to tell whether a failure came from reading the email wrong or judging it wrong.

**Choosing fields by where they go downstream:**

| Field | Downstream use |
|---|---|
| `sender_name`, `sender_email`, `sender_company` | CRM contact match / create |
| `subject`, `is_reply` | Thread linking, deduplication |
| `mentioned_dates` | Deadlines, follow-up tasks |
| `mentioned_dollar_amounts` | Deal value, competitive pricing |
| `questions_asked` | Reply drafting |
| `requested_actions` | Task creation for the rep |

v1 below is the original schema, unchanged.

In [3]:
class ExtractedEmailV1(BaseModel):
    sender_name: Optional[str] = Field(description="Full name of sender if identifiable, else null")
    sender_email: Optional[str] = Field(description="Email address of sender")
    sender_company: Optional[str] = Field(description="Company name if mentioned in signature or domain, else null")
    subject: str = Field(description="Subject line, verbatim")
    is_reply: bool = Field(description="True if subject starts with Re: or body contains quoted prior message")
    mentioned_dates: list[str] = Field(description="Dates mentioned in body, normalized to ISO 8601 (YYYY-MM-DD). Empty list if none.")
    mentioned_dollar_amounts: list[float] = Field(description="Dollar amounts mentioned as floats. Empty list if none.")
    questions_asked: list[str] = Field(description="Literal questions the sender asks, verbatim from the email. Empty list if none.")
    requested_actions: list[str] = Field(description="Concrete actions the sender is asking the recipient to take. Empty list if none.")


FIELDS = list(ExtractedEmailV1.model_fields.keys())

## 4. Prompt v1 and the extraction function
v1 is the original prompt: two few-shot examples, including one for missing data and a forwarded chain. The function takes the prompt, schema, temperature, and reference date as parameters so v1 and v2 run through the same code path.

In [4]:
SYSTEM_PROMPT_V1 = """You extract structured data from sales emails. Follow the schema exactly.

Rules:
- Extract only what is verifiable from the text. Do not infer or guess.
- Normalize dates to ISO 8601 (YYYY-MM-DD). If a date is ambiguous (e.g., "next Tuesday"), skip it.
- Dollar amounts as floats without currency symbols.
- Questions must be verbatim from the email, ending with a question mark.
- If a field cannot be determined from the text, use null (for optional fields) or an empty list.

Example 1:
Input:
---
From: Sarah Chen <sarah@acmecorp.com>
Subject: Pricing question for Q1 rollout

Hi team,

We're evaluating vendors for a rollout starting 2025-03-15. Our budget is around $45,000.
Can you share pricing for 50 seats? Also, do you offer volume discounts?

Please send a proposal by end of week.

Thanks,
Sarah Chen
Acme Corp
---
Output:
{
  "sender_name": "Sarah Chen",
  "sender_email": "sarah@acmecorp.com",
  "sender_company": "Acme Corp",
  "subject": "Pricing question for Q1 rollout",
  "is_reply": false,
  "mentioned_dates": ["2025-03-15"],
  "mentioned_dollar_amounts": [45000.00],
  "questions_asked": ["Can you share pricing for 50 seats?", "Also, do you offer volume discounts?"],
  "requested_actions": ["Send a proposal by end of week"]
}

Example 2 (missing data, forwarded):
Input:
---
From: unknown@gmail.com
Subject: Re: Fwd: quick question

> Original message: what's the deal
saw ur website. call me
---
Output:
{
  "sender_name": null,
  "sender_email": "unknown@gmail.com",
  "sender_company": null,
  "subject": "Re: Fwd: quick question",
  "is_reply": true,
  "mentioned_dates": [],
  "mentioned_dollar_amounts": [],
  "questions_asked": [],
  "requested_actions": ["Call the sender"]
}
"""


def extract_email(email_text, system_prompt, schema, temperature=None, reference_date=None):
    """Returns (extracted dict, usage). temperature=None uses the API default."""
    user_msg = f"Extract from this email:\n---\n{email_text}\n---"
    if reference_date:
        user_msg = f"Email received on: {reference_date}\n\n{user_msg}"

    kwargs = {"temperature": temperature} if temperature is not None else {}
    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        text_format=schema,
        **kwargs,
    )
    return response.output_parsed.model_dump(), response.usage

## 5. Test set design
Four emails, each chosen to stress a different failure mode:

| # | Case | What it tests |
|---|---|---|
| 1 | Clean inquiry | Baseline: every field present and explicit |
| 2 | Forwarded chain, ambiguous | Quoted third-party text, a date with no year, a question in lowercase |
| 3 | Signature-only company, multiple amounts, no ask | Company from the signature, several amounts, empty lists |
| 4 | Reply with an embedded quote | Quoted contract text, a first-name-only sign-off, a request phrased as a question |

**Limitation:** Four emails show *how* the pipeline fails. They are not a statistically meaningful accuracy estimate. The next step is 30 to 50 labeled emails from a real inbox.

In [5]:
TEST_EMAILS = [
    {
        "id": "E1",
        "case": "Clean inquiry",
        "text": """From: Marcus Rivera <m.rivera@northstar.io>
Subject: Interested in your enterprise plan

Hi,

I'm the VP of Ops at Northstar Logistics. We're looking to onboard around 200 users starting April 1, 2025.
Our budget is $85,000 for year one.

What's included in the enterprise tier? Can we schedule a demo next week?

Best,
Marcus""",
    },
    {
        "id": "E2",
        "case": "Forwarded chain, ambiguous",
        "text": """From: jenny@example.com
Subject: Fwd: Re: Fwd: pricing

---------- Forwarded message ----------
From: someone else
> we need to know cost for 30 seats and 100 seats
> also does it integrate with salesforce

can you get back to me by 12/15?""",
    },
    {
        "id": "E3",
        "case": "Signature-only company, multiple amounts",
        "text": """From: David Okafor <d.okafor@meridian-health.org>
Subject: Following up

Circling back on our conversation. We're comparing three vendors — your $12,500 quote, another at $9,800,
and a third at $15,000.

--
David Okafor
Director of IT | Meridian Health
Sent from my iPhone""",
    },
    {
        "id": "E4",
        "case": "Reply with embedded quote",
        "text": """From: priya.s@bluetechlabs.com
Subject: Re: Contract terms

Thanks for sending this over.

> Section 4.2 covers termination with 30 days notice

Can we change that to 60 days notice? Everything else looks fine to sign by 2025-02-28.

Priya""",
    },
]

# The test emails have no Date header. In production the received date comes from the header;
# here a fixed date stands in for it.
REFERENCE_DATE = "2024-12-01"

## 6. Ground truth
Consistency alone can't catch an answer that is wrong the same way every time, so each email gets a hand-labeled correct answer. Labeling forced three product decisions the v1 prompt never made:

1. **Dates with no year** resolve to the next occurrence on or after the received date. Guessing a year is never acceptable. Relative phrases ("next week") are skipped.
2. **Questions vs. actions:** A question asking only for information goes in `questions_asked` only. A question that asks the recipient to *do* something ("Can we schedule a demo?") goes in both fields, because the two fields feed different workflows (reply drafting vs. task creation).
3. **Quoted or forwarded text is not the sender's own content.** Questions and asks inside `>` quotes are excluded.

**How each field is scored:** Names, emails, subjects, and companies are compared exactly after normalizing case and whitespace. Dates and amounts are compared as sets. Questions must match verbatim as a set. Actions are free text, so each expected action lists acceptable keywords, and the action counts must match.

In [6]:
GROUND_TRUTH = {
    "E1": {
        "sender_name": "Marcus Rivera",
        "sender_email": "m.rivera@northstar.io",
        "sender_company": "Northstar Logistics",
        "subject": "Interested in your enterprise plan",
        "is_reply": False,
        "mentioned_dates": ["2025-04-01"],
        "mentioned_dollar_amounts": [85000.0],
        "questions_asked": ["What's included in the enterprise tier?", "Can we schedule a demo next week?"],
        "requested_actions": [["demo"]],
    },
    "E2": {
        "sender_name": None,
        "sender_email": "jenny@example.com",
        "sender_company": None,
        "subject": "Fwd: Re: Fwd: pricing",
        "is_reply": True,
        "mentioned_dates": ["2024-12-15"],
        "mentioned_dollar_amounts": [],
        "questions_asked": ["can you get back to me by 12/15?"],
        "requested_actions": [["get back", "respond", "reply", "follow up"]],
    },
    "E3": {
        "sender_name": "David Okafor",
        "sender_email": "d.okafor@meridian-health.org",
        "sender_company": "Meridian Health",
        "subject": "Following up",
        "is_reply": False,
        "mentioned_dates": [],
        "mentioned_dollar_amounts": [12500.0, 9800.0, 15000.0],
        "questions_asked": [],
        "requested_actions": [],
    },
    "E4": {
        "sender_name": "Priya",
        "sender_email": "priya.s@bluetechlabs.com",
        "sender_company": "Bluetech Labs",
        "subject": "Re: Contract terms",
        "is_reply": True,
        "mentioned_dates": ["2025-02-28"],
        "mentioned_dollar_amounts": [],
        "questions_asked": ["Can we change that to 60 days notice?"],
        "requested_actions": [["60"]],
    },
}


def _norm(s):
    if s is None:
        return None
    s = s.replace("\u2019", "'").strip().lower()
    return re.sub(r"\s+", " ", s)


def _norm_company(s):
    return None if s is None else re.sub(r"[^a-z0-9]", "", s.lower())


def field_correct(field, expected, actual):
    if field == "sender_company":
        return _norm_company(expected) == _norm_company(actual)
    if field in ("sender_name", "sender_email", "subject"):
        return _norm(expected) == _norm(actual)
    if field == "is_reply":
        return expected == actual
    if field == "mentioned_dates":
        return sorted(expected) == sorted(actual)
    if field == "mentioned_dollar_amounts":
        return sorted(round(x, 2) for x in expected) == sorted(round(x, 2) for x in actual)
    if field == "questions_asked":
        return sorted(_norm(q) for q in expected) == sorted(_norm(q) for q in actual)
    if field == "requested_actions":
        if len(expected) != len(actual):
            return False
        remaining = [_norm(a) for a in actual]
        for keywords in expected:
            hit = next((a for a in remaining if any(k in a for k in keywords)), None)
            if hit is None:
                return False
            remaining.remove(hit)
        return True
    raise ValueError(field)

## 7. Evaluation harness
Each email runs `N_RUNS` times per version. Two measures:

- **Consistency:** Did every run return the same output?
- **Accuracy:** Share of runs where each field matched the ground truth.

In [7]:
def run_version(label, system_prompt, schema, temperature=None, reference_date=None, n=N_RUNS):
    records = []
    for email in TEST_EMAILS:
        runs = []
        for _ in range(n):
            output, usage = extract_email(email["text"], system_prompt, schema, temperature, reference_date)
            runs.append({"output": output, "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens})
        records.append({"id": email["id"], "case": email["case"], "runs": runs})
    return {"label": label, "records": records}


def score_version(result):
    rows = []
    for rec in result["records"]:
        gt = GROUND_TRUTH[rec["id"]]
        outputs = [r["output"] for r in rec["runs"]]
        row = {"email": rec["id"], "case": rec["case"], "consistent": all(o == outputs[0] for o in outputs[1:])}
        for f in FIELDS:
            row[f] = mean(field_correct(f, gt[f], o[f]) for o in outputs)
        row["overall"] = mean(row[f] for f in FIELDS)
        rows.append(row)
    return pd.DataFrame(rows).set_index("email")


def show_scores(result):
    df = score_version(result)
    display(Markdown(f"### {result['label']}"))
    display(df.style.format({f: "{:.0%}" for f in FIELDS + ["overall"]}))
    return df


def show_misses(result):
    """Every field that missed ground truth, with expected vs. one example of what came back."""
    print(f"Misses: {result['label']}\n" + "-" * 60)
    any_miss = False
    for rec in result["records"]:
        gt = GROUND_TRUTH[rec["id"]]
        outputs = [r["output"] for r in rec["runs"]]
        for f in FIELDS:
            wrong = [o[f] for o in outputs if not field_correct(f, gt[f], o[f])]
            if wrong:
                any_miss = True
                print(f"{rec['id']} · {f}  ({len(wrong)}/{len(outputs)} runs)")
                print(f"   expected: {gt[f]}")
                print(f"   got:      {wrong[0]}")
    if not any_miss:
        print("No misses.")

### 7a. Baseline: v1 (original prompt, default temperature, no received date)

In [8]:
v1 = run_version("v1: original prompt, default temperature", SYSTEM_PROMPT_V1, ExtractedEmailV1)
v1_scores = show_scores(v1)
show_misses(v1)

### v1: original prompt, default temperature

,case,consistent,sender_name,sender_email,sender_company,subject,is_reply,mentioned_dates,mentioned_dollar_amounts,questions_asked,requested_actions,overall
email,,,,,,,,,,,,
E1,Clean inquiry,True,100%,100%,100%,100%,100%,100%,100%,100%,0%,89%
E2,"Forwarded chain, ambiguous",False,100%,100%,100%,100%,100%,0%,100%,33%,67%,78%
E3,"Signature-only company, multiple amounts",True,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%
E4,Reply with embedded quote,True,0%,100%,100%,100%,100%,100%,100%,100%,0%,78%


Misses: v1: original prompt, default temperature
------------------------------------------------------------
E1 · requested_actions  (3/3 runs)
   expected: [['demo']]
   got:      []
E2 · mentioned_dates  (3/3 runs)
   expected: ['2024-12-15']
   got:      []
E2 · questions_asked  (2/3 runs)
   expected: ['can you get back to me by 12/15?']
   got:      []
E2 · requested_actions  (1/3 runs)
   expected: [['get back', 'respond', 'reply', 'follow up']]
   got:      []
E4 · sender_name  (3/3 runs)
   expected: Priya
   got:      None
E4 · requested_actions  (3/3 runs)
   expected: [['60']]
   got:      []


### 7b. Failure analysis (from the original v1 run)

| Email | Field | What happened | Root cause |
|---|---|---|---|
| E2 | `mentioned_dates` | "12/15" became `2023-12-15` | The prompt says to skip ambiguous dates, but the model treats "12/15" as unambiguous and fills in a year from its own priors. It has no received date to anchor on. |
| E2 | all | Output drifted between runs; the exported JSON disagreed with the printed output | Default temperature plus an underspecified input. When the rules don't settle the answer, sampling decides it. |
| E2 | `questions_asked` | "can you get back to me by 12/15?" was missed | Most likely lowercase, sitting under a forwarded block. The prompt never says quoted text is excluded, so the boundary is fuzzy. |
| E1 | `requested_actions` | "Can we schedule a demo next week?" not captured as an action | No rule on questions vs. actions, so the model put it in exactly one field. |
| E4 | `requested_actions` | The request to change the notice period to 60 days was missed | Same cause: a request phrased as a question. |
| E4 | `sender_name` | Returned `null` despite the "Priya" sign-off | The schema says "full name," so a first name alone was treated as not identifiable. |

**Note on the rerun:** The invented year (`2023-12-15`) came from the original run. In the rerun above, v1 returned an empty date list for E2 instead. So across sessions, the same input produced three different answers: `2023-12-15`, `"12/15"` left inside an action, and no date at all. That's the drift problem.

**Takeaway:** Only one of these is randomness. The rest are **spec gaps**: the prompt and schema never made the decisions the ground truth requires. Setting temperature to 0 fixes the drift. The rules fix everything else.

## 8. Fixes: v2

| Failure | Fix |
|---|---|
| Drift | `temperature=0` (reduces variance; not a determinism guarantee) |
| Invented year | Pass the received date; year-less dates resolve against it; relative phrases are skipped |
| Missed question in a forwarded chain | Explicit rule: quoted/forwarded (`>`) text is excluded; the sender's own questions count regardless of capitalization |
| Actions phrased as questions | Explicit rule for questions vs. actions, reflected in the few-shot example |
| First-name sign-off | Schema: a first name alone is acceptable |

In [9]:
class ExtractedEmailV2(BaseModel):
    sender_name: Optional[str] = Field(description="Sender's name as written in the From line or sign-off. A first name alone is acceptable. Null if no name appears.")
    sender_email: Optional[str] = Field(description="Email address of sender")
    sender_company: Optional[str] = Field(description="Company named in the signature or body. If none, derive from a non-generic email domain (not gmail.com, example.com, etc.). Null otherwise.")
    subject: str = Field(description="Subject line, verbatim")
    is_reply: bool = Field(description="True if the subject starts with Re: or Fwd:, or the body contains a quoted or forwarded prior message")
    mentioned_dates: list[str] = Field(description="Explicit calendar dates in the sender's own text, as YYYY-MM-DD. If the year is missing, use the next occurrence on or after the received date. Skip relative expressions like 'next week'. Empty list if none.")
    mentioned_dollar_amounts: list[float] = Field(description="Dollar amounts in the sender's own text, as floats. Empty list if none.")
    questions_asked: list[str] = Field(description="Questions written by the sender (not quoted or forwarded text), verbatim, including the question mark. Leave dates inside the question as written. Empty list if none.")
    requested_actions: list[str] = Field(description="Things the sender asks the recipient to do, as short imperative phrases. Include requests phrased as questions. Exclude information-only questions and actions the sender will take themselves. Empty list if none.")


SYSTEM_PROMPT_V2 = """You extract structured data from sales emails. Follow the schema exactly.
The user message includes the date the email was received.

Rules:
- Extract only what is verifiable from the text. Never guess.
- Only the sender's own text counts. Ignore quoted or forwarded content (lines starting with ">", or text below a "Forwarded message" header that was written by someone else).
- Dates: normalize explicit calendar dates to YYYY-MM-DD. If a date has no year, use the next occurrence on or after the received date. Never infer a year any other way. Skip relative expressions ("next week", "Tuesday").
- Dollar amounts as floats without currency symbols.
- questions_asked: every question the sender writes, verbatim, including the question mark, regardless of capitalization. Do not rewrite dates inside questions.
- requested_actions: what the sender asks the recipient to do, as short imperative phrases.
  - A question that asks the recipient to do something ("Can we schedule a call?") goes in BOTH questions_asked and requested_actions.
  - A question that only asks for information ("What does it cost?") goes in questions_asked only.
- sender_name: a first name alone is acceptable.
- If a field cannot be determined, use null (optional fields) or an empty list.

Example 1:
Input:
---
Email received on: 2025-01-06

From: Sarah Chen <sarah@acmecorp.com>
Subject: Pricing question for Q1 rollout

Hi team,

We're evaluating vendors for a rollout starting 2025-03-15. Our budget is around $45,000.
What does pricing look like for 50 seats? Also, do you offer volume discounts? Could we set up a call on 1/20?

Please send a proposal by end of week.

Thanks,
Sarah Chen
Acme Corp
---
Output:
{
  "sender_name": "Sarah Chen",
  "sender_email": "sarah@acmecorp.com",
  "sender_company": "Acme Corp",
  "subject": "Pricing question for Q1 rollout",
  "is_reply": false,
  "mentioned_dates": ["2025-03-15", "2025-01-20"],
  "mentioned_dollar_amounts": [45000.00],
  "questions_asked": ["What does pricing look like for 50 seats?", "Also, do you offer volume discounts?", "Could we set up a call on 1/20?"],
  "requested_actions": ["Set up a call on 2025-01-20", "Send a proposal by end of week"]
}

Example 2 (missing data, forwarded):
Input:
---
Email received on: 2025-01-06

From: unknown@gmail.com
Subject: Re: Fwd: quick question

> Original message: what's the deal? can you send a quote?
saw ur website. call me
---
Output:
{
  "sender_name": null,
  "sender_email": "unknown@gmail.com",
  "sender_company": null,
  "subject": "Re: Fwd: quick question",
  "is_reply": true,
  "mentioned_dates": [],
  "mentioned_dollar_amounts": [],
  "questions_asked": [],
  "requested_actions": ["Call the sender"]
}
"""

In [10]:
v2 = run_version(
    "v2: tightened prompt + schema, temperature=0, received date",
    SYSTEM_PROMPT_V2,
    ExtractedEmailV2,
    temperature=0,
    reference_date=REFERENCE_DATE,
)
v2_scores = show_scores(v2)
show_misses(v2)

### v2: tightened prompt + schema, temperature=0, received date

,case,consistent,sender_name,sender_email,sender_company,subject,is_reply,mentioned_dates,mentioned_dollar_amounts,questions_asked,requested_actions,overall
email,,,,,,,,,,,,
E1,Clean inquiry,True,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%
E2,"Forwarded chain, ambiguous",True,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%
E3,"Signature-only company, multiple amounts",True,100%,100%,100%,100%,100%,100%,100%,100%,100%,100%
E4,Reply with embedded quote,True,100%,100%,0%,100%,100%,100%,100%,100%,0%,78%


Misses: v2: tightened prompt + schema, temperature=0, received date
------------------------------------------------------------
E4 · sender_company  (3/3 runs)
   expected: Bluetech Labs
   got:      bluetechlabs.com
E4 · requested_actions  (3/3 runs)
   expected: [['60']]
   got:      []


### 8b. What v2 didn't fix

| Email | Field | What happened | Why | Next fix |
|---|---|---|---|---|
| E4 | `sender_company` | **Regression:** returned `bluetechlabs.com`; v1 had `Bluetech Labs` | The v2 schema said to "derive from a non-generic email domain." The model derived the domain itself, not a company name. My wording caused this. | Describe the output, not the source: "Company name in title case. If only a domain is available, convert it to a name (bluetechlabs.com → Bluetech Labs)." |
| E4 | `requested_actions` | Still missed the request to move the notice period to 60 days | Even with the explicit rule, the model reads "Can we change that…" as negotiation, not a task. The rule's example ("Can we schedule a call?") is too far from a contract change to generalize. | Add a few-shot example with a contract/terms change request. |

**Takeaway:** Fixing one field's spec can break another. That's why this runs as a regression suite with per-field scoring, not a single overall number. Without the per-field table, the 86% → 94% gain would have hidden the company regression.

## 9. Before / after

In [11]:
comparison = pd.DataFrame({
    "v1 accuracy": v1_scores[FIELDS].mean(),
    "v2 accuracy": v2_scores[FIELDS].mean(),
})
comparison["change"] = comparison["v2 accuracy"] - comparison["v1 accuracy"]
comparison.loc["OVERALL"] = [v1_scores["overall"].mean(), v2_scores["overall"].mean(),
                             v2_scores["overall"].mean() - v1_scores["overall"].mean()]
display(comparison.style.format("{:+.0%}", subset=["change"]).format("{:.0%}", subset=["v1 accuracy", "v2 accuracy"]))

consistency = pd.DataFrame({
    "v1 consistent": v1_scores["consistent"],
    "v2 consistent": v2_scores["consistent"],
})
display(consistency)

display(Markdown(
    f"**Overall field accuracy:** {v1_scores['overall'].mean():.0%} → {v2_scores['overall'].mean():.0%}  \n"
    f"**Emails fully consistent across {N_RUNS} runs:** {int(v1_scores['consistent'].sum())}/{len(TEST_EMAILS)} → "
    f"{int(v2_scores['consistent'].sum())}/{len(TEST_EMAILS)}  \n"
    f"**Emails with every field correct on every run:** {int((v1_scores['overall'] == 1).sum())}/{len(TEST_EMAILS)} → "
    f"{int((v2_scores['overall'] == 1).sum())}/{len(TEST_EMAILS)}"
))

,v1 accuracy,v2 accuracy,change
sender_name,75%,100%,+25%
sender_email,100%,100%,+0%
sender_company,100%,75%,-25%
subject,100%,100%,+0%
is_reply,100%,100%,+0%
mentioned_dates,75%,100%,+25%
mentioned_dollar_amounts,100%,100%,+0%
questions_asked,83%,100%,+17%
requested_actions,42%,75%,+33%
OVERALL,86%,94%,+8%


,v1 consistent,v2 consistent
email,,
E1,True,True
E2,False,True
E3,True,True
E4,True,True


**Overall field accuracy:** 86% → 94%  
**Emails fully consistent across 3 runs:** 3/4 → 4/4  
**Emails with every field correct on every run:** 1/4 → 3/4

## 10. Cost per email
v2's longer prompt costs more input tokens. Worth knowing before this runs on every inbound email.

In [12]:
# USD per 1M tokens for MODEL. Confirm against current OpenAI pricing before quoting these numbers.
PRICE_PER_1M = {"input": 2.50, "output": 10.00}


def cost_row(result):
    calls = [r for rec in result["records"] for r in rec["runs"]]
    avg_in = mean(c["input_tokens"] for c in calls)
    avg_out = mean(c["output_tokens"] for c in calls)
    per_email = avg_in / 1e6 * PRICE_PER_1M["input"] + avg_out / 1e6 * PRICE_PER_1M["output"]
    return {"avg input tokens": round(avg_in), "avg output tokens": round(avg_out),
            "cost / email": per_email, "cost / 10k emails": per_email * 10_000}


costs = pd.DataFrame({"v1": cost_row(v1), "v2": cost_row(v2)}).T
display(costs.style.format({"cost / email": "${:.4f}", "cost / 10k emails": "${:,.2f}"}))

,avg input tokens,avg output tokens,cost / email,cost / 10k emails
v1,876,77,$0.0030,$29.62
v2,1236,84,$0.0039,$39.31


## 11. Product implications

**Where each field should go, based on this eval:**

| Tier | Fields | Why |
|---|---|---|
| Auto-write to CRM | `sender_email`, `subject`, `is_reply`, `mentioned_dollar_amounts` | Lifted verbatim or pattern-based; stable in both versions |
| Auto-write, flag if empty | `sender_name`, `sender_company` | Correct when present; a null should prompt a lookup, not a guess |
| Human confirms before acting | `mentioned_dates`, `requested_actions` | These create deadlines and tasks. A wrong date (the invented year) creates a task for the wrong day, which costs more than a missing one. |
| Draft-assist only | `questions_asked` | Feeds reply drafting, where a person already reviews the output |

**Cost vs. accuracy:** v2 costs about 31% more per email (roughly $30 → $39 per 10k emails at the assumed list price), for +8 points of overall accuracy and full consistency. Worth it: one wrong deadline or missed task costs more than the extra tokens across thousands of emails.

**Principles this surfaced:**

1. **Most "model errors" were spec gaps.** Writing ground truth forced decisions (year-less dates, questions vs. actions) that the prompt had left to the model. Label first, then write the prompt.
2. **Consistency is necessary, not sufficient.** Four emails passed a consistency check while still being wrong. Every eval needs a ground truth.
3. **Every prompt change is a regression risk.** The v2 company rule broke a field that v1 got right. Per-field scoring caught it; an overall score would have hidden it.
4. **Match the tolerance for errors to the cost of the error.** An invented date is worse than a missing date, so the prompt skips rather than guesses, and dates stay behind human review.

## 12. Next steps

1. Ship v3 with the two fixes from Section 8b and re-run this harness.
2. Grow the test set to 30 to 50 real, anonymized inbox emails, weighted toward forwarded chains and replies.
3. Add the subjective pass (urgency, intent) as a separate call with its own labeled set.
4. Track per-field accuracy on every prompt change, as a regression suite rather than a one-time check.
5. Run the same harness against a second model to compare accuracy and cost per field.

## 13. Export
Exports the first v2 run for each email, so the file matches the results above.

In [13]:
from google.colab import files

final_output = [rec["runs"][0]["output"] for rec in v2["records"]]

with open("email_extractions.json", "w") as f:
    json.dump(final_output, f, indent=2)

files.download("email_extractions.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>